# NeuroPath AI : entra?nement GPU sur Colab

1. Ouvre ce notebook dans Google Colab, puis s?lectionne **Runtime / Ex?cution > Change runtime type / Modifier le type d'ex?cution > GPU**.
2. Dans la configuration, indique o? se trouve `neuropath-code.zip` dans **Mon Drive**. Le chemin par d?faut correspond ? un ZIP plac? directement dans Mon Drive.
3. Ex?cute les cellules dans l'ordre et autorise la connexion ? Drive.
4. La cellule **Dataset Kaggle** t?l?charge les images automatiquement dans Colab. Tu n'as pas besoin de t?l?charger le dataset sur ton PC ni de le mettre dans Drive.
5. Les checkpoints sont enregistr?s dans `Mon Drive/NeuroPathAI/<RUN_NAME>/results`.

Le dataset Kaggle fourni ne contient pas de correspondance patient v?rifi?e. Ce notebook est donc configur? pour un **essai avec s?paration par image** : `SPLIT_MODE='image'`. Les r?sultats ne mesurent pas la g?n?ralisation ? des patients ind?pendants. Pour utiliser de vrais IDs patients, s?lectionne le mode patient et fournis le CSV et sa provenance.

Le mod?le utilise uniquement des images 2D. Le pr?traitement est partag? avec l'API et l'augmentation est limit?e aux images d'entra?nement.

R?f?rences : [Colab](https://research.google.com/colaboratory/faq.html), [KaggleHub](https://github.com/Kaggle/kagglehub#download-dataset).


In [ ]:
# Keep RUN_NAME unchanged when resuming; use a new name for a new experiment.
RUN_NAME = 'image2d-run1'
CODE_ZIP_DRIVE = '/content/drive/MyDrive/neuropath-code.zip'  # Change si le ZIP est dans un sous-dossier.
RESUME = False
EPOCHS = 30                 # Total target epochs, including saved epochs.
BATCH_SIZE = 32
SEED = 42
PATIENCE = 6
LEARNING_RATE = 0.001

SPLIT_MODE = 'image'      # 'patient' or 'image'
PATIENT_ID_SOURCE = ''      # For patient mode: document the source of genuine IDs.
ALLOW_IMAGE_LEVEL_EXPLORATION = True  # Set True only for exploratory image mode.
DATASET_HANDLE = 'uraninjo/augmented-alzheimer-mri-dataset'

assert SPLIT_MODE in {'patient', 'image'}
assert RUN_NAME and '/' not in RUN_NAME and '\\' not in RUN_NAME and RUN_NAME not in {'.', '..'}
if SPLIT_MODE == 'patient':
    assert PATIENT_ID_SOURCE.strip(), 'Supply genuine patient metadata provenance, or explicitly select exploratory image mode.'
else:
    assert ALLOW_IMAGE_LEVEL_EXPLORATION, 'Image mode requires ALLOW_IMAGE_LEVEL_EXPLORATION=True; patient separation cannot be verified.'


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, hashlib, shutil, subprocess, sys
from zipfile import ZipFile

drive.mount('/content/drive')
experiment = Path('/content/drive/MyDrive/NeuroPathAI') / RUN_NAME
experiment.mkdir(parents=True, exist_ok=True)
run = experiment / 'results'
if RESUME:
    assert (run / 'last_model.pt').is_file(), 'No completed-epoch checkpoint found for this run.'
else:
    assert not run.exists(), 'Results already exist: enable RESUME or choose a new RUN_NAME.'

# Lire le ZIP d?j? pr?sent dans Drive, sans nouvel upload.
code_zip = Path(CODE_ZIP_DRIVE)
if not code_zip.is_file():
    raise FileNotFoundError(
        f'ZIP introuvable : {code_zip}. Dans le panneau Fichiers de Colab, '
        'ouvre drive/MyDrive, trouve le ZIP puis clic droit > Copier le chemin. '
        'Colle ce chemin dans CODE_ZIP_DRIVE et relance cette cellule.'
    )
local_zip = Path('/content/neuropath-code.zip')
shutil.copy2(code_zip, local_zip)
print('Code charg? depuis Drive :', code_zip)
project = Path('/content/neuropath-training')
project.mkdir(exist_ok=True)
with ZipFile(local_zip) as archive:
    for item in archive.infolist():
        assert (project / item.filename).resolve().is_relative_to(project.resolve()), 'Unsafe archive path'
    archive.extractall(project)
if not RESUME:
    shutil.copy2(local_zip, experiment / 'source-code.zip')
%cd /content/neuropath-training
%pip install -r requirements-training.txt


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU. This notebook refuses CPU training.'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## Dataset Kaggle : connexion automatique

La ligne `DATASET_HANDLE = 'uraninjo/augmented-alzheimer-mri-dataset'` identifie d?j? ton dataset. La cellule suivante t?l?charge son contenu **directement sur la machine Colab**, puis trouve le dossier `OriginalDataset` automatiquement.

Tu n'as pas de chemin local ? fournir et tu ne dois pas utiliser le lien Kaggle complet avec les param?tres `fbclid`.

Si Kaggle demande une authentification (401/403), ajoute une cellule contenant `import kagglehub; kagglehub.login()`, suis la connexion demand?e, puis relance le t?l?chargement. Ne mets pas ton token dans une cellule enregistr?e. Si un message indique qu'il faut accepter les conditions du dataset, ouvre sa page Kaggle avec ton compte et accepte-les avant de r?essayer.


In [ ]:
import kagglehub
# If authentication is required, run kagglehub.login() interactively; do not paste tokens into saved cells.
# Dataset images live on the cloud VM; only checkpoints and small reports go to Drive.
provenance_path = experiment / 'dataset.json'
if RESUME:
    provenance = json.loads(provenance_path.read_text())
    handle = provenance['versioned_handle']
else:
    handle = DATASET_HANDLE

print('T?l?chargement du dataset Kaggle dans Colab :', handle)
dataset = Path(kagglehub.dataset_download(handle))
originals = list(dataset.rglob('OriginalDataset'))
assert len(originals) == 1, f'Expected exactly one OriginalDataset folder, found {originals}'
originals = originals[0]
if not RESUME:
    # KaggleHub's normal cache layout ends in versions/<number>.
    version = dataset.name
    assert version.isdigit(), 'Cannot resolve dataset version; use an explicit /versions/N handle and inspect the download path.'
    versioned_handle = DATASET_HANDLE.split('/versions/')[0] + '/versions/' + version
    provenance_path.write_text(json.dumps({'versioned_handle': versioned_handle}, indent=2))
print('Original images:', originals)
print('Pinned dataset:', json.loads(provenance_path.read_text()))


In [ ]:
patient_manifest = None
if SPLIT_MODE == 'patient':
    patient_manifest = experiment / 'patients.csv'
    if not RESUME:
        print('Upload patients.csv with path,patient_id columns; paths relative to OriginalDataset.')
        mapping_upload = files.upload()
        assert 'patients.csv' in mapping_upload
        patient_manifest.write_bytes(mapping_upload['patients.csv'])
    assert patient_manifest.is_file()
else:
    print('EXPLORATORY IMAGE SPLIT: patient overlap and near-duplicate leakage remain unverified.')


In [ ]:
command = [sys.executable, '-u', 'scripts/train_images.py',
           '--data-dir', str(originals), '--output-dir', str(run),
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
           '--seed', str(SEED), '--patience', str(PATIENCE),
           '--lr', str(LEARNING_RATE), '--require-cuda']
if patient_manifest is not None:
    command += ['--patient-manifest', str(patient_manifest), '--patient-id-source', PATIENT_ID_SOURCE]
else:
    command += ['--allow-image-split']
if RESUME:
    command += ['--resume']
print('Checkpoints persist at:', run)
subprocess.run(command, check=True)


## Resume after a disconnect
Reconnect to a GPU runtime, reopen this notebook, set `RESUME=True`, and keep the same run name, code bundle in Drive, split settings and hyperparameters. Rerun the cells. The pinned dataset version is downloaded again, and training resumes after the last completed epoch.

`last_model.pt` stores weights, optimizer state, completed epoch, early-stopping state, history and random-number-generator states. `best_model.pt` stores the best validation model plus preprocessing and class metadata for inference. An unfinished epoch is repeated after interruption. Resume refuses changes to image hashes, patient mapping, split membership, training code or core hyperparameters. You may increase the total epoch target; early stopping still applies.

Runtime versions and installed packages are recorded. Exact bitwise reproducibility is not guaranteed across different hardware or software versions. Google Drive access requires your Colab authorization; this notebook has not been executed in your cloud account.


In [ ]:
metrics = json.loads((run / 'test_metrics.json').read_text())
print(json.dumps(metrics, indent=2))
print('Keep the test split held out: do not select models by these test results.')
print('Persistent best checkpoint:', run / 'best_model.pt')
bundle = shutil.make_archive('/content/neuropath-training-results', 'zip', run)
files.download(bundle)
# Extract locally and set NEUROPATH_MODEL to the downloaded best_model.pt.
# Clinical/genetic inputs are not used by this 2D model; full 3D scans are rejected.
